1) I have pdf files in a folder. The program reads PDF files and stores it in vectorDB

2) The user asks a question, the vector DB return relevant chunks related to the query

3) I pass the retrieved results + query + prompt to the LLM

Here I am using OpenAI embedding model to create embeddings

In [1]:
# Check python version: Should be 3.10+
!python --version

Python 3.11.9


In [ ]:
# # This took 5 minutes

# !pip install langchain==1.3.6 langchain-community==0.4.2 langchain-openai==1.2.2 langchain-text-splitters==1.1.2 

In [ ]:
# !pip install pypdf==6.10.2

In [ ]:
# !pip install pinecone==7.3.0
# !pip install python-dotenv

# Now restart the kernel

In [1]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", 
           "langchain-text-splitters", "pypdf", "pinecone"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
langchain-text-splitters version: 1.1.2
pypdf version: 6.10.2
pinecone version: 7.3.0


In [2]:
# import Libraries

import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from pinecone import Pinecone, ServerlessSpec

C:\Users\hi\AppData\Local\Temp\ipykernel_26888\916316018.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [11]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")

EMBEDDING_DIM=1536
PINECONE_INDEX = 'loxford-quantum-mechanics'

DOCUMENT_DIR = "documents-0"

# Phase 1 - Ingestion: Populate the Vector DB

In [4]:
## Read the document
def read_doc(directory):
    file_loader=PyPDFDirectoryLoader(directory)
    documents=file_loader.load()
    return documents

docs=read_doc(DOCUMENT_DIR)
print("Number of pages:",len(docs))
print("Type of object:", type(docs[0]))

Number of pages: 2
Type of object: <class 'langchain_core.documents.base.Document'>


In [5]:
# Show me pages
import os

for doc in docs:
    file_path = doc.metadata.get("source", "")
    file_name = os.path.basename(file_path)
    page = doc.metadata.get("page")
    
    print("File:",file_name)
    print("Page:", page)
    print("First few snippet:", doc.page_content[:200])
    print("---------")

File: Quantum Mechanics-LoxfordAcademy-small.pdf
Page: 0
First few snippet: Quantum Mechanics: A Grand Tour of the Tiny Universe 
Executive Summary: Quantum mechanics is the branch of physics governing the very 
small – atoms, electrons, photons, and other particles. As one s
---------
File: Quantum Mechanics-LoxfordAcademy-small.pdf
Page: 1
First few snippet: Loxford Academy: 
Loxford Academy was founded by Alberto Graciaso Negurasa and Rohit Salvadore Lomax. 
Its address is 5432, Deru Valley on planet Mars.  Loxford Academy has 34,000 employees 
and has a
---------


In [8]:
# Split pdf into pages and split pages into chunks

def chunk_data(docs, chunk_size=800, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = []

    for doc in docs:
        chunks = splitter.split_text(doc.page_content)

        for chunk in chunks:
            chunked_docs.append({
                "text": chunk,
                "metadata": {
                    "source": doc.metadata.get("source", ""),
                    "page": doc.metadata.get("page")
                }
            })

    return chunked_docs

documents=chunk_data(docs=docs)
print("Total chunks =", len(documents))
# print(documents[0])

for d in documents:
    print("-- text:", d['text'])
    print("-- metadata:", d['metadata'])
    print("+++++++++++++++++++++++++++++++")

Total chunks = 4
-- text: Quantum Mechanics: A Grand Tour of the Tiny Universe 
Executive Summary: Quantum mechanics is the branch of physics governing the very 
small – atoms, electrons, photons, and other particles. As one source notes, “Quantum 
mechanics is the fundamental physical theory that describes the behavior of matter and of 
light; its unusual characteristics typically occur…at and below the scale of atoms.”[1]. In 
other words, it’s how the universe works when you zoom in far beyond everyday 
experience. It arose in the early 1900s to explain puzzling experiments (like blackbody 
radiation and the photoelectric effect) that classical physics couldn’t[2]. This report gives a 
concise introduction: we’ll define quantum mechanics, outline its history with a timeline,
-- metadata: {'source': 'documents-0\\Quantum Mechanics-LoxfordAcademy-small.pdf', 'page': 0}
+++++++++++++++++++++++++++++++
-- text: concise introduction: we’ll define quantum mechanics, outline its history wi

## Insert the above embedding chunks to Pinecone vectorDB

### Create index

In [12]:
## Vector Search DB In Pinecone
pc = Pinecone(
        api_key=PINECONE_API_KEY
    )


if PINECONE_INDEX not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_INDEX,
        dimension=EMBEDDING_DIM,
        metric='euclidean',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )


index = pc.Index(PINECONE_INDEX)
print("Index Stats:\n", index.describe_index_stats())

============== {'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'euclidean',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


### Use OpenAI embedding model
- I could have used some other embedding models too.

In [13]:
## Embedding Technique Of OPENAI
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", # cheapest model
    api_key=OPENAI_API_KEY
)

print("Embedding model=", embeddings.model)
print("Embeddings=",embeddings) # info about embedding

Embedding model= text-embedding-3-small
Embeddings= client=<openai.resources.embeddings.Embeddings object at 0x000001A25C68A690> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001A25C68AD10> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


In [14]:
def get_embedding(text):
    embedding = embeddings.embed_query(doc.page_content)
    return embedding


In [14]:
# Lets see how embedding looks like
sample_text = "Hello world. Today it is my first day at Loxford."

# Generate the embedding
sample_embedding = embeddings.embed_query(sample_text)

# Print the visual characteristics
print(f"Text: '{sample_text}'")
print(f"Embedding Type: {type(sample_embedding)}")
print(f"Vector Dimensions (Length): {len(sample_embedding)}")
print(f"First 5 Values: {sample_embedding[:5]}")


Text: 'Hello world. Today it is my first day at Loxford.'
Embedding Type: <class 'list'>
Vector Dimensions (Length): 1536
First 5 Values: [-0.0187835693359375, -0.0093841552734375, -0.013702392578125, -0.0181427001953125, 0.01303863525390625]


In [15]:
def populate_vectorDB(documents):

    for i, doc in enumerate(documents):

        # Extract text and metadata
        text = doc.get("text", "")
        metadata = doc.get("metadata", {})

        file_path = metadata.get("source", "")
        file_name = os.path.basename(file_path)
        page_number = metadata.get("page", 0)

        # Generate embedding
        embedding = embeddings.embed_query(text)

        # Insert into Pinecone
        index.upsert([{
            "id": str(i),
            "values": embedding,
            "metadata": {
                "text": text,
                "source": file_name,
                "page": page_number,
                "chunk_id": i
            }
        }])


# call the function
populate_vectorDB(documents)
print("Documents inserted into vector DB")

Documents inserted into vector DB


# (OPTIONAL) Delete Pinecone Index

In [24]:
pc.delete_index(PINECONE_INDEX)

In [25]:
for idx in pc.list_indexes():
    print(idx)